# Fraud Detection — CRISP-DM Notebook

**IS-455 · Chapter 17 Assignment**

This notebook follows the six CRISP-DM phases to build a fraud-detection classifier that predicts `is_fraud` on e-commerce orders. The trained model is saved to `jobs/model.pkl` and consumed by `jobs/run_inference.py`, which scores every unfulfilled order and writes `fraud_probability` into `order_predictions` so the warehouse priority queue can surface high-risk orders.

---
**Phases covered:**
1. Business Understanding
2. Data Understanding
3. Data Preparation
4. Modeling
5. Evaluation
6. Deployment

In [ ]:
import sqlite3
import pathlib
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

DB_PATH   = pathlib.Path('shop.db')
JOBS_DIR  = pathlib.Path('jobs')
JOBS_DIR.mkdir(exist_ok=True)
MODEL_PATH = JOBS_DIR / 'model.sav'

print('DB exists:', DB_PATH.exists())

# Load orders joined with customer profile — no shipments table needed
con = sqlite3.connect(DB_PATH)

df = pd.read_sql_query("""
    SELECT
        o.order_id,
        o.payment_method,
        o.device_type,
        o.ip_country,
        o.promo_used,
        o.order_total,
        o.risk_score,
        o.is_fraud,
        c.customer_segment,
        c.loyalty_tier
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
""", con)

con.close()

print(f'Loaded {len(df):,} orders with {df.columns.tolist()}')
df.head()

## Phase 1 — Business Understanding

**Problem:** Online retailers lose revenue to fraudulent orders — chargebacks, stolen goods, and manual review costs. Early detection at order time lets the warehouse hold suspicious shipments before they leave the building.

**Business goal:** Flag orders with a high probability of being fraudulent (`is_fraud = 1`) so that the warehouse priority queue can route them to a review lane instead of picking/packing immediately.

**ML goal:** Train a binary classifier that predicts `is_fraud` using features known at the moment an order is placed (payment method, device type, geography, order value, customer profile). The model must produce a **probability score** so that the warehouse team can set their own risk threshold.

**Success criteria:**
- F1-score ≥ 0.70 on the held-out test set (fraud class)
- ROC-AUC ≥ 0.80
- Precision high enough to avoid flooding the review queue with false positives

**Deployment path (Ch. 17 pipeline):**
`shop.db` → notebook trains → `jobs/model.pkl` → `run_inference.py` scores pending orders → `order_predictions` table → warehouse queue UI

## Phase 2 — Data Understanding

In [ ]:
import sqlite3
import pathlib
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

DB_PATH   = pathlib.Path('shop.db')
JOBS_DIR  = pathlib.Path('jobs')
JOBS_DIR.mkdir(exist_ok=True)
MODEL_PATH = JOBS_DIR / 'model.pkl'

print('DB exists:', DB_PATH.exists())

# Load orders joined with customer profile — no shipments table needed
con = sqlite3.connect(DB_PATH)

df = pd.read_sql_query("""
    SELECT
        o.order_id,
        o.payment_method,
        o.device_type,
        o.ip_country,
        o.promo_used,
        o.order_total,
        o.risk_score,
        o.is_fraud,
        c.customer_segment,
        c.loyalty_tier
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
""", con)

con.close()

print(f'Loaded {len(df):,} orders with {df.columns.tolist()}')
df.head()

In [ ]:
# ── Target distribution ───────────────────────────────────────────────────────
print('--- Target distribution ---')
print(df['is_fraud'].value_counts())
print()
print(df['is_fraud'].value_counts(normalize=True).round(3).rename({0: 'legitimate', 1: 'fraud'}))

print('\n--- Fraud rate by payment method ---')
print(df.groupby('payment_method')['is_fraud'].mean().round(3).sort_values(ascending=False))

print('\n--- Fraud rate by ip_country ---')
print(df.groupby('ip_country')['is_fraud'].mean().round(3).sort_values(ascending=False))

print('\n--- Fraud rate by device_type ---')
print(df.groupby('device_type')['is_fraud'].mean().round(3).sort_values(ascending=False))

print('\n--- Numeric feature summary (fraud vs legitimate) ---')
print(df.groupby('is_fraud')[['order_total', 'risk_score']].mean().round(2))

print('\n--- Nulls ---')
print(df.isnull().sum())

### Discover Relationships (Ch. 8)
Correlation analysis and cross-tabulations to understand how features relate to fraud.

In [ ]:
# ── Numeric correlations with is_fraud ───────────────────────────────────────
print('--- Pearson correlations with is_fraud ---')
print(df[['order_total', 'promo_used', 'risk_score', 'is_fraud']].corr()['is_fraud'].drop('is_fraud').round(3))

print('\n--- Fraud rate by payment_method ---')
print(df.groupby('payment_method')['is_fraud'].agg(['mean','sum','count']).round(3).sort_values('mean', ascending=False))

print('\n--- Fraud rate by loyalty_tier ---')
print(df.groupby('loyalty_tier')['is_fraud'].agg(['mean','sum','count']).round(3).sort_values('mean', ascending=False))

print('\n--- Fraud rate by customer_segment ---')
print(df.groupby('customer_segment')['is_fraud'].agg(['mean','sum','count']).round(3).sort_values('mean', ascending=False))

# Average risk_score for fraud vs legitimate
print('\n--- risk_score stats by fraud label ---')
print(df.groupby('is_fraud')['risk_score'].describe().round(2))

## Phase 3 — Data Preparation

Key observations from Phase 2:
- `is_fraud` is **imbalanced** (~5–15% fraud). We use `class_weight='balanced'` to compensate.
- `risk_score` and `payment_method` are the strongest signals.
- `ip_country` is dropped — too many categories with sparse counts; `risk_score` already captures the geographic signal.
- All features are known at order placement time — no look-ahead leakage.

In [ ]:
df['customer_segment'] = df['customer_segment'].fillna('unknown')
df['loyalty_tier']     = df['loyalty_tier'].fillna('none')

CATEGORICAL_FEATURES = [
    'payment_method',
    'device_type',
    'customer_segment',
    'loyalty_tier',
]

NUMERIC_FEATURES = [
    'order_total',
    'promo_used',
    'risk_score',
]

FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES
TARGET   = 'is_fraud'

X = df[FEATURES]
y = df[TARGET]

print('Features:', FEATURES)
print('X shape:', X.shape)
print(f'Class balance — fraud: {y.mean():.1%}  legitimate: {1 - y.mean():.1%}')

## Phase 4 — Modeling

We use a `scikit-learn Pipeline` (Ch. 13–14) that chains:
1. `ColumnTransformer` — `OrdinalEncoder` for categoricals, passthrough for numerics
2. `RandomForestClassifier` — robust to non-linear patterns and feature scaling; `class_weight='balanced'` corrects for fraud imbalance

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {len(X_train):,}  Test: {len(X_test):,}')
print(f'Fraud in train: {y_train.mean():.1%}  Test: {y_test.mean():.1%}')

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), CATEGORICAL_FEATURES),
    ('num', 'passthrough', NUMERIC_FEATURES),
])

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=5,
        class_weight='balanced',   # handles fraud imbalance
        random_state=42,
        n_jobs=-1,
    )),
])

model.fit(X_train, y_train)
print('Training complete.')

### Model Tuning (Ch. 15)
Use `GridSearchCV` with 3-fold cross-validation to find the best hyperparameters. We search over `max_depth` and `n_estimators` — the two parameters with the most impact on RandomForest performance vs. training time.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Build the pipeline with placeholder params (tuning will override them)
tune_pipeline = Pipeline(steps=[
    ('preprocessor', ColumnTransformer(transformers=[
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), CATEGORICAL_FEATURES),
        ('num', 'passthrough', NUMERIC_FEATURES),
    ])),
    ('classifier', RandomForestClassifier(
        class_weight='balanced', random_state=42, n_jobs=-1
    )),
])

param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth':    [6, 8, 10],
    'classifier__min_samples_leaf': [5, 10],
}

grid_search = GridSearchCV(
    tune_pipeline,
    param_grid,
    cv=3,
    scoring='f1',          # optimise for fraud F1
    n_jobs=-1,
    verbose=0,
)
grid_search.fit(X_train, y_train)

print('Best params:', grid_search.best_params_)
print(f'Best CV F1:  {grid_search.best_score_:.4f}')

# Use the best estimator going forward
model = grid_search.best_estimator_
print('Model updated to best estimator.')

## Phase 5 — Evaluation

For fraud detection, **accuracy alone is misleading** because of class imbalance. We report:
- **Precision** — of orders flagged as fraud, how many were actually fraud (false positive cost)
- **Recall** — of all actual fraud orders, how many did we catch (false negative cost)
- **F1-score** — harmonic mean, balances precision and recall
- **ROC-AUC** — model's ability to rank fraudulent orders above legitimate ones

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['legitimate', 'fraud']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')

print('\n--- Confusion Matrix ---')
cm = confusion_matrix(y_test, y_pred)
print(pd.DataFrame(cm,
    index=['actual: legitimate', 'actual: fraud'],
    columns=['predicted: legitimate', 'predicted: fraud']
))

In [ ]:
# ── Feature importances ───────────────────────────────────────────────────────
importances = model.named_steps['classifier'].feature_importances_
feat_df = pd.Series(importances, index=FEATURES).sort_values(ascending=False)
print('--- Feature importances ---')
print(feat_df.round(4))

### Feature Selection (Ch. 16)
Identify which features contribute meaningfully. Features with near-zero importance are candidates for removal in a leaner production model.

In [ ]:
from sklearn.feature_selection import SelectFromModel

# SelectFromModel uses the trained forest's feature importances
# to identify features above the mean importance threshold
selector = SelectFromModel(model.named_steps['classifier'], prefit=True, threshold='mean')

# Get which features were selected (after preprocessing, indices map back to FEATURES)
selected_mask = selector.get_support()
selected_features = [f for f, keep in zip(FEATURES, selected_mask) if keep]
dropped_features  = [f for f, keep in zip(FEATURES, selected_mask) if not keep]

print('Selected features (above mean importance):', selected_features)
print('Dropped  features (below mean importance):', dropped_features)
print(f'\n{len(selected_features)} of {len(FEATURES)} features selected.')
print('All features are kept in the final model to maximise recall on a small dataset.')

## Phase 6 — Deployment

The trained pipeline is serialized with `joblib` to `jobs/model.pkl`. The inference script `jobs/run_inference.py` loads this file, queries all unfulfilled orders from `shop.db`, and writes `fraud_probability` + `predicted_fraud` into the `order_predictions` table. The web app's "Run Scoring" button triggers this script, and the warehouse queue joins on `order_predictions` to surface high-risk orders.

In [ ]:
joblib.dump(model, MODEL_PATH)
print(f'Model saved to {MODEL_PATH}')

In [ ]:
# ── Smoke test: reload model and score a sample ───────────────────────────────
loaded_model = joblib.load(MODEL_PATH)

sample = X_test.iloc[:5].copy()
probs  = loaded_model.predict_proba(sample)[:, 1]
preds  = loaded_model.predict(sample)

sample = sample.copy()
sample['fraud_probability']  = probs.round(4)
sample['predicted_fraud']    = preds
sample['actual_fraud']       = y_test.iloc[:5].values

print(sample[['fraud_probability', 'predicted_fraud', 'actual_fraud']])
print('\nSmoke test passed — model is ready for run_inference.py')